In [7]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import pyarrow.dataset as ds
from pathlib import Path
from pyspark.sql import SparkSession, functions as F, Window
from scipy.stats import spearmanr
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import KFold, GroupKFold, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

spark = (SparkSession.builder.appName("fraud").master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "64")
         .config("spark.sql.execution.arrow.pyspark.enabled", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("WARN")

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
TABLES = ROOT / "tables"
OUT = ROOT / "data" / "curated"; OUT.mkdir(parents=True, exist_ok=True)

def read_part(name):
    p = TABLES / name
    return (spark.read.option("basePath", str(p)).parquet(str(p))
            .withColumn("order_datetime", F.to_date("order_datetime")))

raw = {p: read_part(p) for p in ["part_2", "part_3", "part_4"]}
all_tx = raw["part_2"].unionByName(raw["part_3"], allowMissingColumns=True) \
                      .unionByName(raw["part_4"], allowMissingColumns=True)
tx = all_tx.filter(F.col("dollar_value") >= 0.01)      
print(f"{all_tx.count():,} transactions loaded")

KeyboardInterrupt: 

In [2]:
ENT = {"consumer": {"key": "user_id",      "other": "merchant_abn", "file": "consumer_fraud_probability.csv"},
       "merchant": {"key": "merchant_abn", "other": "user_id",      "file": "merchant_fraud_probability.csv"}}

FEATS = ["n_txn", "total_value", "avg_value", "max_value", "n_other", "txn_per_other",
         "days_active", "prior_days", "prior_avg_daily_value", "prior_avg_daily_txn",
         "value_vs_prior", "txn_vs_prior", "dow", "month", "days_since_start"]

def load_labels(file, key):
    df = spark.read.option("header", True).option("inferSchema", True).csv(str(TABLES / "part_1" / file))
    return (df.withColumn("order_datetime", F.to_date("order_datetime"))
              .groupBy(key, "order_datetime").agg(F.avg("fraud_probability").alias("fraud_probability")))

def build_daily(key, other):
    d = tx.groupBy(key, "order_datetime").agg(
        F.count("*").alias("n_txn"), F.sum("dollar_value").alias("total_value"),
        F.avg("dollar_value").alias("avg_value"), F.max("dollar_value").alias("max_value"),
        F.countDistinct(other).alias("n_other"))
    w_hist = Window.partitionBy(key).orderBy("order_datetime").rowsBetween(Window.unboundedPreceding, -1)
    w_all = Window.partitionBy(key)
    return (d
        .withColumn("txn_per_other", F.col("n_txn") / F.col("n_other"))
        .withColumn("days_active", F.datediff("order_datetime", F.min("order_datetime").over(w_all)))
        .withColumn("prior_days", F.count(F.lit(1)).over(w_hist))                   
        .withColumn("prior_avg_daily_value", F.avg("total_value").over(w_hist))
        .withColumn("prior_avg_daily_txn", F.avg("n_txn").over(w_hist))
        .withColumn("value_vs_prior", F.col("total_value") / F.col("prior_avg_daily_value"))
        .withColumn("txn_vs_prior", F.col("n_txn") / F.col("prior_avg_daily_txn"))
        .withColumn("dow", F.dayofweek("order_datetime"))
        .withColumn("month", F.month("order_datetime"))
        .withColumn("days_since_start", F.datediff("order_datetime", F.lit("2021-02-28").cast("date"))))

def make_model(n_rows):
    return HistGradientBoostingRegressor(
        max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
        min_samples_leaf=min(50, max(5, n_rows // 20)),      
        l2_regularization=1.0, early_stopping=(n_rows >= 1000), random_state=42)

def scores(actual, pred):
    pred = np.clip(pred, 0, 100)
    return {"MAE": mean_absolute_error(actual, pred),
            "RMSE": mean_squared_error(actual, pred) ** 0.5,
            "R2": r2_score(actual, pred),
            "Spearman": spearmanr(actual, pred).statistic if np.ptp(pred) > 0 else np.nan}

def report(train, valid, title):
    model = make_model(len(train)).fit(train[FEATS], train["fraud_probability"])
    y = valid["fraud_probability"].to_numpy()
    base = np.full(len(valid), train["fraud_probability"].mean())
    print(title, f"({len(train):,} train / {len(valid):,} test)")
    print(pd.DataFrame({"mean baseline": scores(y, base), "model": scores(y, model.predict(valid[FEATS]))}).T.round(3), "\n")

In [4]:
def build_daily(key, other):
    return (tx.groupBy(key, "order_datetime").agg(
                F.count("*").alias("n_txn"), F.sum("dollar_value").alias("total_value"),
                F.avg("dollar_value").alias("avg_value"), F.max("dollar_value").alias("max_value"),
                F.approx_count_distinct(other).alias("n_other"))
            .withColumn("txn_per_other", F.col("n_txn") / F.col("n_other"))
            .withColumn("dow", F.dayofweek("order_datetime"))
            .withColumn("month", F.month("order_datetime"))
            .withColumn("days_since_start", F.datediff("order_datetime", F.lit("2021-02-28").cast("date"))))

FEATS = ["n_txn", "total_value", "avg_value", "max_value", "n_other", "txn_per_other",
         "dow", "month", "days_since_start"]

In [6]:
train_pd = {}
for name, c in ENT.items():
    key = c["key"]
    path = OUT / f"{name}_day_features"
    build_daily(key, c["other"]).write.mode("overwrite").parquet(str(path))
    daily = spark.read.parquet(str(path))
    lab = load_labels(c["file"], key)
    df = daily.join(lab, [key, "order_datetime"], "inner").toPandas()
    df["order_datetime"] = pd.to_datetime(df["order_datetime"])
    train_pd[name] = df.sort_values("order_datetime").reset_index(drop=True)
    print(f"{name}: {len(df):,} labelled days, {df[key].nunique():,} distinct {key}s, "
          f"{daily.count():,} days to score in total")

NameError: name 'OUT' is not defined

In [5]:
for name in ENT:
    df = train_pd[name].sample(frac=1, random_state=42)    
    k = int(len(df) * 0.8)                                  
    report(df.iloc[:k], df.iloc[k:], f"{name.upper()} - 80/20 split")

NameError: name 'train_pd' is not defined

In [9]:
c = ENT["consumer"]; key = c["key"]; col = "predicted_consumer_fraud_probability"
df = train_pd["consumer"]
final_model = make_model(len(df)).fit(df[FEATS], df["fraud_probability"])    

parts = []
for batch in ds.dataset(str(OUT / "consumer_day_features"), format="parquet").to_batches(batch_size=500_000):
    chunk = batch.to_pandas()
    chunk[col] = np.clip(final_model.predict(chunk[FEATS]), 0, 100).round(2)
    parts.append(chunk[[key, "order_datetime", col]])
pred = pd.concat(parts, ignore_index=True)
pred_path = OUT / "consumer_day_fraud_predictions.parquet"
pred.to_parquet(pred_path, index=False)
print(f"{len(pred):,} consumer-days scored | nulls: {int(pred[col].isna().sum())}")
print(pred[col].quantile([0.01, 0.5, 0.9, 0.99, 0.999]).round(2).to_dict())

8,976,568 consumer-days scored | nulls: 0
{0.01: 8.89, 0.5: 8.89, 0.9: 8.89, 0.99: 9.34, 0.999: 22.66}


In [ ]:
known = load_labels(c["file"], key).withColumnRenamed("fraud_probability", "known_consumer_fraud_probability")
pred_sp = spark.read.parquet(str(pred_path)).withColumn("order_datetime", F.to_date("order_datetime"))

for part, df_raw in raw.items():
    out = (df_raw.join(pred_sp, [key, "order_datetime"], "left")
                 .join(known, [key, "order_datetime"], "left")
                 .withColumn("consumer_fraud_probability_final",
                             F.coalesce("known_consumer_fraud_probability", col)))
    out.write.mode("overwrite").parquet(str(OUT / f"{part}_fraud_scored"))
    chk = spark.read.parquet(str(OUT / f"{part}_fraud_scored"))
    print(part, f"{chk.count():,} rows | null scores: "
                f"{chk.filter(F.col('consumer_fraud_probability_final').isNull()).count():,}")

NameError: name 'OUT' is not defined

In [ ]:
scored = spark.read.parquet(*[str(OUT / f"{p}_fraud_scored") for p in raw])
q99 = scored.approxQuantile(col, [0.99], 0.001)[0]          # high-risk day = top 1% of scores

mr = (scored.groupBy("merchant_abn").agg(
        F.count("*").alias("n_txn"),
        F.avg("consumer_fraud_probability_final").alias("avg_consumer_risk"),
        F.avg((F.col(col) >= q99).cast("int")).alias("high_risk_share"))
      .toPandas())
mr = mr[mr["n_txn"] >= 30].copy()                           
for m in ["avg_consumer_risk", "high_risk_share"]:
    mr[m + "_pct"] = mr[m].rank(pct=True) * 100
mr["merchant_risk_score"] = mr[["avg_consumer_risk_pct", "high_risk_share_pct"]].mean(axis=1).round(1)
mr.to_parquet(OUT / "merchant_risk_scores.parquet", index=False)


kn = (load_labels(ENT["merchant"]["file"], "merchant_abn").groupBy("merchant_abn")
      .agg(F.avg("fraud_probability").alias("known_fraud")).toPandas())
chk = mr.merge(kn, on="merchant_abn")
print(f"{len(mr):,} merchants scored | {len(chk)} have known labels")
print("Spearman vs known merchant fraud:", round(spearmanr(chk["merchant_risk_score"], chk["known_fraud"]).statistic, 3))
mr.nlargest(10, "merchant_risk_score")[["merchant_abn", "n_txn", "avg_consumer_risk", "high_risk_share", "merchant_risk_score"]]